# Khám phá 1 incident RCAEval

Mục tiêu: **hiểu dữ liệu** của một incident trước khi thiết kế grouping/ranking. Notebook chỉ **đọc** raw, không sửa raw/Parquet, không train model.

Các bước:

| Bước | Làm gì | Câu hỏi cần trả lời |
| --- | --- | --- |
| 0 | Chọn case | Case nào, hệ thống nào? |
| 1 | Đọc + kiểm tra | Dữ liệu có đủ, đúng khoảng thời gian không? |
| 2 | Tổng quan | Container nào nhiều log? Trước/sau lỗi thay đổi ra sao? |
| 3 | Template | Từ N log còn bao nhiêu mẫu (template)? |
| 4 | Top template | Mỗi container thường in gì? |
| 5 | So sánh trước/sau | Template nào mới xuất hiện, tăng, giảm? |
| 6 | Mở nhãn | Service gây lỗi có dấu hiệu gì trong log? |
| 7 | Data card | Ghi lại quan sát |

**Chế độ "blind":** bước 1–5 không in tên case (vì tên thư mục chứa đáp án, ví dụ `re2ob_checkoutservice_cpu_1`). Bạn tự quan sát và đoán trước, đến bước 6 mới mở nhãn. Cách này giúp tránh "nhìn đáp án rồi mới thấy dấu hiệu".

**Lưu ý leakage:** những case đã khám phá ở đây phải nằm trong **tập train** khi chia split, vì bạn đã nhìn nhãn của chúng. Bước 7 có thể ghi case vào `explored_cases.csv` để nhớ.

Môi trường: chạy bằng `.venv` của `Resource/explore_datasets/` (cần `pandas`, `pyarrow`). Không cần GPU.

## 0. Cấu hình và chọn case

In [ ]:
from pathlib import Path
import hashlib
import sys

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def find_repo_root(start: Path) -> Path:
    """Đi ngược lên tới thư mục chứa AGENTS.md (root repo)."""
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "Resource").exists():
            return path
    raise FileNotFoundError("Không tìm thấy root repo. Hãy mở notebook từ trong repo NCKH.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAW_ROOT = REPO_ROOT / "NCKH_rawData" / "raw_data" / "huggingface" / "rcaeval"
OUTPUT_ROOT = REPO_ROOT / "Resource" / "notebooks" / "explore_output"

# Dùng đúng hàm normalize đang được adapter RCAEval sử dụng.
sys.path.insert(0, str(REPO_ROOT / "Resource" / "explore_datasets"))
from src.shared.infrastructure.normalization import normalize_message  # noqa: E402

# ---- Chỉnh ở đây ----
CASE_ID = None          # None = chọn ngẫu nhiên theo SYSTEM/SEED; hoặc ghi rõ, ví dụ "re2ob_checkoutservice_cpu_1"
SYSTEM = "ob"           # ob = Online Boutique, ss = Sock Shop, tt = Train Ticket
SUITE = "RE2"           # RE2: lỗi tài nguyên/mạng (cpu, mem, disk, delay, loss, socket); RE3: lỗi code (f1..f5)
SEED = 0
BLIND = True            # True = ẩn tên case và nhãn tới bước 6
TOP_N = 20              # số template in ra mỗi container
MIN_COUNT = 5           # ngưỡng tối thiểu để coi một template là "xuất hiện"
SAVE = False            # True = bước 7 ghi data card + explored_cases.csv vào OUTPUT_ROOT

print("Repo:", REPO_ROOT)
print("Raw RCAEval tồn tại:", RAW_ROOT.exists())

In [ ]:
cases = pd.read_parquet(RAW_ROOT / "cases.parquet")
eligible = cases[(cases["system"] == SYSTEM) & (cases["suite"] == SUITE) & (cases["has_logs"])]
print(f"Case có log cho {SUITE}-{SYSTEM.upper()}: {len(eligible)} / {len(cases)} case trong cases.parquet")

if CASE_ID is None:
    CASE_ID = eligible.sample(1, random_state=SEED)["case"].iloc[0]
case_row = cases.set_index("case").loc[CASE_ID]
CASE_DIR = RAW_ROOT / CASE_ID
if not (CASE_DIR / "logs.parquet").exists():
    raise FileNotFoundError(f"Case không có logs.parquet: {CASE_DIR}")

# Cột chứa đáp án: chỉ mở ở bước 6.
LABEL_COLUMNS = ["case", "root_cause_service", "fault", "fault_description", "has_root_cause_file"]
public_info = case_row.drop([c for c in LABEL_COLUMNS if c in case_row.index])
print("Case:", "<ẩn tới bước 6>" if BLIND else CASE_ID)
public_info[["dataset", "system_name", "time_start", "time_end", "inject_time",
             "duration_minutes", "n_logs", "has_traces"]]

## 1. Đọc log và kiểm tra cơ bản

`logs.parquet` của RCAEval có 3 cột: `timestamp` (epoch **giây**), `container_name`, `message`. Không có cột mức độ (severity).
`inject_time.txt` là thời điểm bắt đầu tiêm lỗi.

In [ ]:
if case_row["n_logs"] > 3_000_000:
    print("Cảnh báo: case lớn, đọc toàn bộ có thể tốn nhiều RAM.")

logs = pd.read_parquet(CASE_DIR / "logs.parquet")
inject_time = int((CASE_DIR / "inject_time.txt").read_text().strip())
# log_id tạm cho notebook: vị trí dòng trong file nguồn (giữ để truy ngược raw).
logs = logs.reset_index().rename(columns={"index": "source_row"})

checks = {
    "số dòng": len(logs),
    "khớp n_logs trong cases.parquet": len(logs) == int(case_row["n_logs"]),
    "timestamp null": int(logs["timestamp"].isna().sum()),
    "container null": int(logs["container_name"].isna().sum()),
    "message null/rỗng": int((logs["message"].isna() | (logs["message"].str.strip() == "")).sum()),
    "timestamp min": int(logs["timestamp"].min()),
    "timestamp max": int(logs["timestamp"].max()),
    "inject_time (file)": inject_time,
    "inject_time khớp cases.parquet": inject_time == int(case_row["inject_time"]),
    "inject nằm trong khoảng log": logs["timestamp"].min() <= inject_time <= logs["timestamp"].max(),
}
pd.Series(checks, name="giá trị").to_frame()

## 2. Tổng quan theo container

Chia log thành **trước** (`timestamp < inject_time`) và **sau** lỗi. Hai cửa sổ có thể dài khác nhau, nên so sánh bằng **số log mỗi phút**.

In [ ]:
logs["phase"] = (logs["timestamp"] >= inject_time).map({False: "before", True: "after"})
minutes = {
    "before": max((inject_time - logs["timestamp"].min()) / 60, 1e-9),
    "after": max((logs["timestamp"].max() + 1 - inject_time) / 60, 1e-9),
}
print({k: round(v, 1) for k, v in minutes.items()}, "phút")

overview = logs.pivot_table(index="container_name", columns="phase", values="source_row",
                            aggfunc="count", fill_value=0).reindex(columns=["before", "after"], fill_value=0)
overview["total"] = overview["before"] + overview["after"]
overview["rate_before/min"] = (overview["before"] / minutes["before"]).round(1)
overview["rate_after/min"] = (overview["after"] / minutes["after"]).round(1)
# +1 để tránh chia cho 0 khi container không có log trước lỗi.
overview["rate_change"] = ((overview["rate_after/min"] + 1) / (overview["rate_before/min"] + 1)).round(2)
overview.sort_values("total", ascending=False)

### Timeline theo phút

Mỗi dòng là một phút tính từ lúc tiêm lỗi (âm = trước lỗi). Nhìn xem container nào thay đổi số log quanh mốc 0.

In [ ]:
logs["minute_from_inject"] = (logs["timestamp"] - inject_time) // 60
timeline = logs.pivot_table(index="minute_from_inject", columns="container_name", values="source_row",
                            aggfunc="count", fill_value=0)
timeline.loc[-5:5]   # 5 phút trước và sau mốc tiêm lỗi; bỏ .loc để xem toàn bộ

## 3. Template: gom log giống nhau

Dùng `normalize_message` có sẵn trong code (che timestamp, UUID, hex, số…), rồi MD5 12 ký tự làm `template_id`, giống adapter RCAEval hiện tại.

**Đây là regex masking, chưa phải Drain3/Spell.** Masking quá mạnh có thể gộp nhầm hai log khác nghĩa; quá yếu thì mỗi log thành một template.

In [ ]:
unique_messages = logs["message"].drop_duplicates()
template_of = {m: normalize_message(m) for m in unique_messages}
logs["template"] = logs["message"].map(template_of)
logs["template_id"] = logs["template"].map(lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:12])

n_logs = len(logs)
n_messages = logs["message"].nunique()
n_templates = logs["template_id"].nunique()
print(f"{n_logs:,} log → {n_messages:,} message khác nhau → {n_templates:,} template")
print(f"Tỷ lệ giảm số mục ở mức template (toàn incident): {1 - n_templates / n_logs:.4%}")

per_container = logs.groupby("container_name").agg(
    logs=("source_row", "count"), templates=("template_id", "nunique"))
per_container["reduction"] = (1 - per_container["templates"] / per_container["logs"]).round(4)

# Kiểm tra truy ngược: mỗi dòng log thuộc đúng một template, không mất dòng nào.
assert logs["template_id"].notna().all() and len(logs) == n_logs
per_container.sort_values("logs", ascending=False)

## 4. Top template mỗi container

Mỗi bảng: template, số lần trước/sau lỗi, và một message gốc làm ví dụ. Đọc để hiểu **container này bình thường in gì**.

In [ ]:
counts = logs.pivot_table(index=["container_name", "template_id"], columns="phase", values="source_row",
                          aggfunc="count", fill_value=0).reindex(columns=["before", "after"], fill_value=0)
counts["total"] = counts["before"] + counts["after"]
example = logs.drop_duplicates(["container_name", "template_id"]).set_index(["container_name", "template_id"])
counts = counts.join(example[["template", "message"]].rename(columns={"message": "example_message"}))
counts = counts.reset_index()

for container in overview.sort_values("total", ascending=False).index:
    top = counts[counts["container_name"] == container].nlargest(TOP_N, "total")
    print(f"\n=== {container}: {per_container.loc[container, 'templates']} template, top {len(top)} ===")
    display(top[["template_id", "before", "after", "template"]].reset_index(drop=True))

## 5. So sánh trước / sau lỗi

Ba loại thay đổi theo template (chỉ tính template có ít nhất `MIN_COUNT` lần):

- **new**: chỉ xuất hiện sau lỗi.
- **gone**: có trước lỗi, biến mất sau lỗi.
- **changed**: tần suất mỗi phút thay đổi mạnh nhất.

Thêm một bảng đếm từ khóa lỗi (`error`, `fail`, `exception`, `timeout`, `refused`…). **Đây là heuristic tìm chữ trong message**, không phải severity của log: chữ "error" có thể nằm trong nội dung bình thường.

In [ ]:
counts["rate_before"] = counts["before"] / minutes["before"]
counts["rate_after"] = counts["after"] / minutes["after"]
counts["change"] = ((counts["rate_after"] + 0.1) / (counts["rate_before"] + 0.1)).round(2)

new_t = counts[(counts["before"] == 0) & (counts["after"] >= MIN_COUNT)]
gone_t = counts[(counts["after"] == 0) & (counts["before"] >= MIN_COUNT)]
changed_t = counts[(counts["before"] > 0) & (counts["after"] > 0) & (counts["total"] >= MIN_COUNT)]

cols = ["container_name", "template_id", "before", "after", "change", "template"]
print(f"Template MỚI sau lỗi: {len(new_t)}")
display(new_t.sort_values("after", ascending=False)[cols].head(30).reset_index(drop=True))
print(f"Template BIẾN MẤT sau lỗi: {len(gone_t)}")
display(gone_t.sort_values("before", ascending=False)[cols].head(30).reset_index(drop=True))
print("Template thay đổi tần suất mạnh nhất (tăng và giảm):")
display(pd.concat([changed_t.nlargest(10, "change"), changed_t.nsmallest(10, "change")])[cols].reset_index(drop=True))

In [ ]:
ERROR_PATTERN = r"(?i)\b(?:error|err|fail(?:ed|ure)?|exception|timeout|timed out|refused|unavailable|panic|fatal)\b"
logs["has_error_word"] = logs["message"].str.contains(ERROR_PATTERN, regex=True, na=False)
error_table = logs[logs["has_error_word"]].pivot_table(
    index="container_name", columns="phase", values="source_row", aggfunc="count", fill_value=0
).reindex(index=overview.index, columns=["before", "after"], fill_value=0)
error_table["rate_change"] = (((error_table["after"] / minutes["after"]) + 0.1)
                              / ((error_table["before"] / minutes["before"]) + 0.1)).round(2)
print(f"Log có từ khóa lỗi: {int(logs['has_error_word'].sum()):,} / {n_logs:,}")
error_table.sort_values("after", ascending=False)

### Dừng lại và đoán trước khi mở nhãn

Ghi vào đây (sửa cell markdown này):

- Container tôi nghi ngờ nhất: …
- Lý do (template mới? tần suất đổi? từ khóa lỗi? container im lặng?): …
- Loại lỗi tôi đoán: …

## 6. Mở nhãn và đối chiếu

Bảng xếp hạng dưới đây dùng **ba tín hiệu đơn giản** để xem service gây lỗi đứng ở đâu. Đây là **quan sát trên một case**, không phải kết quả RCA và không phải baseline đã đánh giá.

- `volume_change`: mức thay đổi số log/phút (lấy độ lệch khỏi 1, tăng hay giảm đều tính).
- `new_templates`: số template mới sau lỗi.
- `error_rate_change`: mức tăng log có từ khóa lỗi.

In [ ]:
root_cause = case_row["root_cause_service"]
print("Case:", CASE_ID)
print("Root cause service:", root_cause)
print("Fault:", case_row["fault"], "-", case_row["fault_description"])
print("Root cause có trong container_name của log:", root_cause in set(logs["container_name"]))

# Không làm tròn trước khi xếp hạng, để tránh tạo đồng hạng giả.
signals = pd.DataFrame({
    "volume_change": ((overview["rate_after/min"] + 1) / (overview["rate_before/min"] + 1) - 1).abs(),
    "new_templates": new_t.groupby("container_name").size(),
    "error_rate_change": error_table["rate_change"],
}).fillna(0)
SIGNALS = list(signals.columns)


def rank_note(col: str, service: str) -> str:
    """Hạng của service theo một tín hiệu, kèm số container đồng hạng; tín hiệu hằng số thì không xếp hạng."""
    if signals[col].nunique() == 1:
        return "không phân biệt (mọi container bằng nhau)"
    rank = int(signals[col].rank(ascending=False, method="min")[service])
    ties = int((signals[col] == signals.loc[service, col]).sum()) - 1
    return f"{rank}/{len(signals)}" + (f" (đồng hạng với {ties})" if ties else "")


for col in SIGNALS:
    signals[f"rank_{col}"] = signals[col].rank(ascending=False, method="min").astype(int)
signals["is_root_cause"] = signals.index == root_cause
if root_cause in signals.index:
    for col in SIGNALS:
        print(f"{col}: root cause hạng {rank_note(col, root_cause)}")
signals.round(3).sort_values("rank_volume_change")

In [ ]:
if root_cause in set(logs["container_name"]):
    rc = counts[counts["container_name"] == root_cause]
    print(f"=== {root_cause}: top template theo mức thay đổi ===")
    display(rc[rc["total"] >= MIN_COUNT].sort_values("change", ascending=False)[cols].head(15).reset_index(drop=True))
    print(f"=== {root_cause}: timeline quanh mốc lỗi ===")
    display(timeline[[root_cause]].loc[-5:5].T)
else:
    print("Service gây lỗi không có log trong case này: chỉ dựa trên log thì không thể chỉ ra nó trực tiếp.")

## 7. Data card

Cell dưới in một data card có số liệu tự động; phần **Quan sát** bạn tự điền. Nếu `SAVE = True`, card được ghi vào `Resource/notebooks/explore_output/` và case được thêm vào `explored_cases.csv` (danh sách case **phải để ở tập train**).

In [ ]:
rank_text = ("service gây lỗi không có log" if root_cause not in signals.index
             else "<br>".join(f"{col}: {rank_note(col, root_cause)}" for col in SIGNALS))
card = f"""# Data card — {CASE_ID}

| Mục | Giá trị |
| --- | --- |
| Dataset / hệ thống | {case_row['dataset']} / {case_row['system_name']} |
| Fault | {case_row['fault']} ({case_row['fault_description']}) |
| Root cause service | {root_cause} |
| Số log / message khác nhau / template | {n_logs:,} / {n_messages:,} / {n_templates:,} |
| Số container | {logs['container_name'].nunique()} |
| Thời lượng trước / sau lỗi (phút) | {minutes['before']:.1f} / {minutes['after']:.1f} |
| Root cause có log | {root_cause in set(logs['container_name'])} |
| Hạng root cause theo 3 tín hiệu đơn giản | {rank_text} |

## Quan sát (tự điền)

- Dấu hiệu của service gây lỗi trong log:
- Service bị ảnh hưởng và dấu hiệu:
- Template bị gộp nhầm / tách thừa do regex:
- Log có đủ để nhận ra lỗi này không? Vì sao:
- Đoán ở bước 5 đúng hay sai:
"""
print(card)

if SAVE:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / f"{CASE_ID}.md").write_text(card, encoding="utf-8")
    registry = OUTPUT_ROOT / "explored_cases.csv"
    seen = pd.read_csv(registry) if registry.exists() else pd.DataFrame(columns=["case"])
    if CASE_ID not in set(seen["case"]):
        pd.concat([seen, pd.DataFrame({"case": [CASE_ID]})]).to_csv(registry, index=False)
    print("Đã ghi:", OUTPUT_ROOT)

## Sau khi chạy xong

Lặp lại với 3–5 case khác: đổi `SEED` hoặc `CASE_ID`, thử các loại lỗi khác (`cpu`, `mem`, `delay`, `loss`, `disk`, `socket`), rồi đổi `SYSTEM = "ss"` hoặc `SUITE = "RE3"`.

Câu hỏi cần trả lời sau 3–5 case:

1. Loại lỗi nào để lại dấu vết rõ trong log, loại nào gần như không?
2. Service gây lỗi thường "ồn" hơn, "im" hơn, hay giống bình thường?
3. Regex template có gộp nhầm/tách thừa không? Có cần parser tốt hơn (Drain3)?
4. Service bị ảnh hưởng (ví dụ `frontend`) có thường nổi bật hơn service gây lỗi không? Nếu có, ranking theo "ồn nhất" sẽ sai.